In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Load dataset with 'latin1' encoding to handle special characters and avoid decode errors
df= pd.read_csv("/content/sample_data/online_retail_II.csv", encoding='latin1')

# EDA

In [ ]:
df.head(6)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom


In [ ]:
df.shape

(1067371, 8)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


CHECKING NULL-VALUES

In [ ]:
df.isnull().sum()

,0
Invoice,0
StockCode,0
Description,4382
Quantity,0
InvoiceDate,0
Price,0
Customer ID,243007
Country,0


# SUMMARY STATISTICS

In [ ]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


DROPPING IRRLEVANT COLUMNS

In [ ]:
# Define a list of columns that are not needed for this analysis
drop_cols = ['StockCode']

In [ ]:
# Remove the specified columns permanently from the dataframe to save memory and clean the data
df.drop(columns=drop_cols, inplace=True)

In [ ]:
# Drop any rows where 'Customer ID' is missing, ensuring we only analyze known customer transactions
df = df.dropna(subset=['Customer ID'])

In [ ]:
# Fill missing product descriptions with 'Unknown' to keep the rows without leaving blank values
df['Description'] = df['Description'].fillna('Unknown')

CREATING NEW COLUMN

In [ ]:
# Create a True/False column to flag transactions as cancellations if the quantity is negative
df['IsCancellation'] = df['Quantity'] < 0

In [ ]:
# Using .loc to safely create a new 'Revenue' column calculated by multiplying quantity by unit price
df.loc[:, 'Revenue'] = df['Quantity'] * df['Price']

In [ ]:
# Categorize transactions as 'Wholesale' if the order size is 100 or more units, otherwise label it 'Retail'
df['OrderType'] = df['Quantity'].apply(
    lambda x: 'Wholesale' if abs(x) >= 100 else 'Retail')

CONVERTING DATE & CREATING TIME AND DAY FEATURES


In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
df['Year'] = df['InvoiceDate'].dt.year

In [ ]:
df['Month'] = df['InvoiceDate'].dt.month

In [ ]:
df['MonthName'] = df['InvoiceDate'].dt.month_name()

In [ ]:
df['YearMonth'] = df['InvoiceDate'].dt.strftime('%Y-%m')

In [ ]:
df['DayName'] = df['InvoiceDate'].dt.day_name()

In [ ]:
df['Hour'] = df['InvoiceDate'].dt.hour

REMOVING INVALID TRANSACTIONS

In [ ]:
sales_df = df[(df['Price'] > 0)].copy()

CREATING NEW SALES TRANSACTIONS TABLE FOR SQL ANALYSIS


In [ ]:
#Creating list
sales_transactions = sales_df[
    [
        'Invoice',
        'Description',
        'Quantity',
        'Price',
        'InvoiceDate',
        'Customer ID',
        'Country',
        'Revenue',
        'OrderType',
        'IsCancellation'
    ]
]

In [ ]:
sales_transactions.head(11)

,Invoice,Description,Quantity,Price,InvoiceDate,Customer ID,Country,Revenue,OrderType,IsCancellation
0,489434,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,6.95,2009-12-01 07:45:00,13085.0,United Kingdom,83.4,Retail,False
1,489434,PINK CHERRY LIGHTS,12,6.75,2009-12-01 07:45:00,13085.0,United Kingdom,81.0,Retail,False
2,489434,WHITE CHERRY LIGHTS,12,6.75,2009-12-01 07:45:00,13085.0,United Kingdom,81.0,Retail,False
3,489434,"RECORD FRAME 7"" SINGLE SIZE",48,2.10,2009-12-01 07:45:00,13085.0,United Kingdom,100.8,Retail,False
4,489434,STRAWBERRY CERAMIC TRINKET BOX,24,1.25,2009-12-01 07:45:00,13085.0,United Kingdom,30.0,Retail,False
5,489434,PINK DOUGHNUT TRINKET POT,24,1.65,2009-12-01 07:45:00,13085.0,United Kingdom,39.6,Retail,False
6,489434,SAVE THE PLANET MUG,24,1.25,2009-12-01 07:45:00,13085.0,United Kingdom,30.0,Retail,False
7,489434,FANCY FONT HOME SWEET HOME DOORMAT,10,5.95,2009-12-01 07:45:00,13085.0,United Kingdom,59.5,Retail,False
8,489435,CAT BOWL,12,2.55,2009-12-01 07:46:00,13085.0,United Kingdom,30.6,Retail,False
9,489435,"DOG BOWL , CHASING BALL DESIGN",12,3.75,2009-12-01 07:46:00,13085.0,United Kingdom,45.0,Retail,False


In [ ]:
sales_transactions.to_csv("SalesTransactions.csv",index=False)

In [ ]:
#from google.colab import files
#files.download('SalesTransactions.csv')